# Lezione 9: reti generative avversarie (GAN)

In questa lezione costruisci e addestri, dal vivo, una rete generativa avversaria (GAN, dall'inglese *Generative Adversarial Network*) minimale. Esegui ogni cella di codice e osservi come il generatore impara a imitare una distribuzione di dati reali.

## Obiettivi di apprendimento

Al termine di questa lezione, sai:

- Spiegare il rapporto tra generatore e discriminatore con l'analogia del falsario e del poliziotto
- Descrivere l'obiettivo minimax che addestra entrambe le reti contemporaneamente
- Implementare in PyTorch un generatore e un discriminatore con livelli lineari
- Addestrare una GAN minimale su dati sintetici bidimensionali
- Visualizzare l'avanzamento dell'addestramento confrontando dati reali e dati generati
- Riconoscere i motivi che rendono instabile l'addestramento di una GAN, tra cui il collasso di modalità

## Il falsario e il poliziotto

Immagina un falsario di banconote e la polizia che cerca di scoprirlo. All'inizio, le banconote false sono grossolane e la polizia le individua senza fatica. Il falsario impara da ogni rifiuto e migliora la carta, l'inchiostro, i dettagli. Anche la polizia impara e affina il proprio giudizio. Col tempo, entrambe le parti migliorano in parallelo.

Questa dinamica è una GAN:

- Il **generatore** fa da falsario: parte da rumore casuale e produce dati che cerca di far passare per reali
- Il **discriminatore** fa da poliziotto: riceve dati reali e dati generati, e decide per ciascuno se è autentico o falso

Nessuna delle due reti vede l'obiettivo finale dell'altra. Il generatore non osserva mai i dati reali: riceve solo il segnale del discriminatore su quanto le sue creazioni reggano. Più il discriminatore distingue bene, più informazioni riceve il generatore su dove sta sbagliando.

## Il gioco minimax

Il generatore e il discriminatore si addestrano contemporaneamente, con obiettivi opposti:

- Il discriminatore vuole **massimizzare** la propria precisione: alta probabilità per i dati reali, bassa probabilità per i dati generati
- Il generatore vuole **minimizzare** quella stessa precisione: vuole che il discriminatore scambi le sue creazioni per dati reali

In pratica, addestri le due reti in modo alternato:

1. Aggiorni il discriminatore perché distingua meglio il reale dal generato
2. Aggiorni il generatore perché inganni meglio il discriminatore
3. Ripeti il ciclo migliaia di volte

Il punto di arrivo ideale è un equilibrio in cui il discriminatore non indovina più di quanto farebbe lanciando una moneta: a quel punto, i dati generati sono statisticamente indistinguibili da quelli reali.

Un dettaglio pratico: invece di minimizzare direttamente la probabilità di essere scoperto (una funzione con gradiente quasi piatto all'inizio dell'addestramento), il generatore usa una variante che massimizza la probabilità che il discriminatore lo classifichi come reale. L'obiettivo punta nella stessa direzione, ma fornisce un gradiente molto più utile fin dal primo passo.

## Costruisci una GAN minimale

Ora costruisci una GAN completa in PyTorch:

- **Dati reali**: punti che formano un cerchio (una miscela di otto gruppi gaussiani), generati con NumPy
- **Generatore**: due livelli nascosti; riceve un vettore di rumore bidimensionale e restituisce un punto bidimensionale
- **Discriminatore**: due livelli nascosti; riceve un punto bidimensionale e restituisce un logit di "realtà"

Esegui la cella successiva per definire i dati, i due modelli, la funzione di perdita e gli ottimizzatori.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# --- Datos reales: mezcla de ocho gaussianas en círculo ---
def sample_real_data(n_samples, n_modes=8, radius=4.0, std=0.15):
    angles = 2 * np.pi * np.random.randint(0, n_modes, size=n_samples) / n_modes
    centers = np.stack([radius * np.cos(angles), radius * np.sin(angles)], axis=1)
    points = centers + std * np.random.randn(n_samples, 2)
    return torch.tensor(points, dtype=torch.float32)


LATENT_DIM = 2


# --- Generador: ruido latente -> punto en dos dimensiones ---
class Generator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, z):
        return self.net(z)


# --- Discriminador: punto en dos dimensiones -> logit de "realidad" ---
class Discriminator(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),  # logit, sin sigmoide: se aplica dentro de la pérdida
        )

    def forward(self, x):
        return self.net(x)


generator = Generator().to(device)
discriminator = Discriminator().to(device)

# BCEWithLogitsLoss combina sigmoide y entropía cruzada binaria de forma numéricamente estable
criterion = nn.BCEWithLogitsLoss()
opt_g = torch.optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))

print(generator)
print(discriminator)

## Osserva l'addestramento dal vivo

Esegui la cella successiva e lasciala girare. Ogni tot passi, il codice aggiorna un diagramma a dispersione con i dati reali (gli otto gruppi disposti in cerchio) e i dati che il generatore produce in quel momento.

Osserva tre cose mentre procede:

- All'inizio, i punti generati formano una nube sfocata senza relazione con il cerchio
- Man mano che l'addestramento avanza, la nube si avvicina agli otto gruppi reali
- Alla fine, verifica se il generatore copre tutti gli otto gruppi o se si concentra solo su due o tre: questo è il collasso di modalità

In [ ]:
BATCH_SIZE = 256
N_STEPS = 4000
PLOT_EVERY = 500

for step in range(1, N_STEPS + 1):
    # ---------- 1) Actualiza el discriminador ----------
    real_data = sample_real_data(BATCH_SIZE).to(device)
    z = torch.randn(BATCH_SIZE, LATENT_DIM, device=device)
    fake_data = generator(z).detach()  # detach: el generador no se actualiza en este paso

    real_labels = torch.ones(BATCH_SIZE, 1, device=device)
    fake_labels = torch.zeros(BATCH_SIZE, 1, device=device)

    loss_d_real = criterion(discriminator(real_data), real_labels)
    loss_d_fake = criterion(discriminator(fake_data), fake_labels)
    loss_d = loss_d_real + loss_d_fake

    opt_d.zero_grad()
    loss_d.backward()
    opt_d.step()

    # ---------- 2) Actualiza el generador (pérdida no saturante) ----------
    z = torch.randn(BATCH_SIZE, LATENT_DIM, device=device)
    fake_data = generator(z)
    # El generador "quiere" que el discriminador clasifique sus muestras como reales
    loss_g = criterion(discriminator(fake_data), real_labels)

    opt_g.zero_grad()
    loss_g.backward()
    opt_g.step()

    # ---------- 3) Dibuja el progreso cada PLOT_EVERY pasos ----------
    if step % PLOT_EVERY == 0 or step == 1:
        with torch.no_grad():
            real_vis = sample_real_data(500).numpy()
            z_vis = torch.randn(500, LATENT_DIM, device=device)
            fake_vis = generator(z_vis).cpu().numpy()

        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.scatter(real_vis[:, 0], real_vis[:, 1], s=10, alpha=0.5, label="Datos reales")
        ax.scatter(fake_vis[:, 0], fake_vis[:, 1], s=10, alpha=0.5, label="Datos generados")
        ax.set_title(f"Paso {step} de {N_STEPS} | pérdida D: {loss_d.item():.3f} | pérdida G: {loss_g.item():.3f}")
        ax.legend(loc="upper right")
        ax.set_xlim(-6, 6)
        ax.set_ylim(-6, 6)
        ax.set_aspect("equal")
        plt.show()

## Perché le GAN sono difficili da addestrare

Addestrare una GAN è più fragile che addestrare una rete che si limita a classificare o prevedere. Questi sono i tre motivi principali:

- **Instabilità del gioco minimax**. Le due reti si aggiornano contemporaneamente inseguendo obiettivi opposti. Se il discriminatore migliora troppo in fretta, lascia il generatore senza un gradiente utile; se migliora troppo lentamente, lascia passare dati di scarsa qualità. Il sistema può oscillare invece di stabilizzarsi in un equilibrio
- **Collasso di modalità**. Il generatore scopre che poche uscite specifiche ingannano bene il discriminatore e smette di esplorare il resto della distribuzione reale. Il risultato è scarsa varietà nei campioni, anche se ciascuno, preso singolarmente, sembra convincente
- **Assenza di una metrica di progresso affidabile**. Trattandosi di un gioco tra due giocatori che migliorano insieme, il valore della perdita non decresce in modo monotono come in un problema di ottimizzazione classico. Non puoi leggere direttamente dalla curva della perdita se il modello sta migliorando o no

La radice teorica di questa fragilità sta nella divergenza che misura l'obiettivo minimax classico. Quando la distribuzione dei dati generati non si sovrappone a quella reale, cosa molto comune all'inizio dell'addestramento, quella divergenza smette di fornire un segnale di apprendimento utile. Soluzioni come la rete generativa avversaria di Wasserstein (WGAN) sostituiscono quella divergenza con una distanza che continua a fornire un gradiente utile anche senza sovrapposizione, il che stabilizza l'addestramento.

## Riepilogo dei punti chiave

- Una GAN combina due reti in competizione: il generatore crea dati e il discriminatore li giudica
- L'obiettivo di addestramento è un gioco minimax: il discriminatore massimizza la propria precisione e il generatore la minimizza
- In pratica, il generatore usa una variante non saturante che fornisce gradienti più utili all'inizio dell'addestramento
- Con livelli lineari semplici e dati sintetici bidimensionali puoi già osservare l'apprendimento e il rischio di collasso di modalità
- L'instabilità dell'addestramento è strutturale, non un errore di implementazione: deriva dalle proprietà della divergenza che il gioco minimax minimizza

**Apri ora il notebook di pratica di accompagnamento `../notebooks_practica/clase09_practica_gan.ipynb` e sperimenta con i parametri dell'addestramento.**